# Customer Churn Intelligence — XGBoost

## Objective

Train **XGBoost** as the main boosted-tree candidate. Perform a small **validation-based** hyperparameter search, evaluate on the validation set, and update the model comparison table. The final test set is **not used**.

**Stage:** Step 13 — XGBoost (no SMOTE, threshold tuning, calibration, SHAP, or test-set evaluation).

### Why XGBoost?
- **Strong for tabular data** — gradient boosted trees are a top choice for structured customer datasets.
- **Captures nonlinear interactions** — learns complex patterns without manual feature crosses.
- **Handles imbalance weighting** — `scale_pos_weight` adjusts for minority churn class.
- **Often stronger than Random Forest** — sequential boosting can refine decision boundaries.

### Why not a neural network?
- This is **relatively small tabular data** (~5k training rows, 19 features).
- Added complexity is **not justified yet** when tree ensembles already perform well (AGENTS.md rule 12).

In [ ]:
import itertools
import sys
from pathlib import Path

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

PROJECT_ROOT = Path("..").resolve()
REPORTS_DIR = PROJECT_ROOT / "reports"
COMPARISON_PATH = REPORTS_DIR / "model_comparison.csv"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_split import load_split_from_manifest
from src.preprocessing import build_preprocessor

RANDOM_STATE = 42

## 1. Load Train / Validation Data

In [ ]:
split = load_split_from_manifest()

X_train, X_val = split.X_train, split.X_val
y_train = (split.y_train == "Yes").astype(int)
y_val = (split.y_val == "Yes").astype(int)

neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
scale_pos_weight_base = round(neg_count / pos_count, 4)

print(f"Train: {len(X_train):,} | Validation: {len(X_val):,}")
print(f"Train class counts — No: {neg_count:,}, Yes: {pos_count:,}")
print(f"scale_pos_weight baseline (neg/pos): {scale_pos_weight_base}")

## 2. Validation-Based Hyperparameter Search

Each candidate is **trained on `X_train` only** and scored on **`X_val`**.  
Selection metric: **F1** (consistent with Random Forest step).  
`scale_pos_weight` grid includes **1.0**, the **train class ratio**, and a slightly higher weight.

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "scale_pos_weight": [1.0, scale_pos_weight_base, round(scale_pos_weight_base * 1.2, 2)],
}

search_rows = []
best_row = None
best_f1 = -1.0
best_pipeline = None

for n_est, depth, lr, sub, col, spw in itertools.product(
    param_grid["n_estimators"],
    param_grid["max_depth"],
    param_grid["learning_rate"],
    param_grid["subsample"],
    param_grid["colsample_bytree"],
    param_grid["scale_pos_weight"],
):
    pipe = Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            (
                "model",
                XGBClassifier(
                    n_estimators=n_est,
                    max_depth=depth,
                    learning_rate=lr,
                    subsample=sub,
                    colsample_bytree=col,
                    scale_pos_weight=spw,
                    random_state=RANDOM_STATE,
                    eval_metric="logloss",
                    n_jobs=-1,
                ),
            ),
        ]
    )
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)
    y_proba = pipe.predict_proba(X_val)[:, 1]

    row = {
        "n_estimators": n_est,
        "max_depth": depth,
        "learning_rate": lr,
        "subsample": sub,
        "colsample_bytree": col,
        "scale_pos_weight": spw,
        "F1": round(f1_score(y_val, y_pred), 4),
        "PR_AUC": round(average_precision_score(y_val, y_proba), 4),
        "ROC_AUC": round(roc_auc_score(y_val, y_proba), 4),
        "Recall": round(recall_score(y_val, y_pred), 4),
        "Precision": round(precision_score(y_val, y_pred, zero_division=0), 4),
    }
    search_rows.append(row)

    if row["F1"] > best_f1:
        best_f1 = row["F1"]
        best_row = row
        best_pipeline = pipe

search_df = pd.DataFrame(search_rows).sort_values("F1", ascending=False)
print(f"Configurations evaluated: {len(search_df)}")
print("\nTop 5 by validation F1:")
search_df.head(5)

In [ ]:
best_params = {
    "n_estimators": best_row["n_estimators"],
    "max_depth": best_row["max_depth"],
    "learning_rate": best_row["learning_rate"],
    "subsample": best_row["subsample"],
    "colsample_bytree": best_row["colsample_bytree"],
    "scale_pos_weight": best_row["scale_pos_weight"],
}
print("Best XGBoost settings (validation F1):")
best_params

## 3. Best Model — Validation Metrics

In [ ]:
y_val_pred = best_pipeline.predict(X_val)
y_val_proba = best_pipeline.predict_proba(X_val)[:, 1]

xgb_metrics = {
    "Model": "XGBoost",
    "Accuracy": round(accuracy_score(y_val, y_val_pred), 4),
    "Precision": round(precision_score(y_val, y_val_pred, zero_division=0), 4),
    "Recall": round(recall_score(y_val, y_val_pred), 4),
    "F1": round(f1_score(y_val, y_val_pred), 4),
    "ROC_AUC": round(roc_auc_score(y_val, y_val_proba), 4),
    "PR_AUC": round(average_precision_score(y_val, y_val_proba), 4),
    "Predicted_Churners": int(y_val_pred.sum()),
}

pd.DataFrame([xgb_metrics])

In [ ]:
cm = confusion_matrix(y_val, y_val_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"],
)
print("Confusion Matrix — XGBoost (best validation config)")
cm_df

## 4. Model Comparison Table

In [ ]:
comparison_df = pd.read_csv(COMPARISON_PATH)
comparison_df = comparison_df[comparison_df["Model"] != "XGBoost"]
comparison_df = pd.concat(
    [comparison_df, pd.DataFrame([{k: v for k, v in xgb_metrics.items() if k != "Predicted_Churners"}])],
    ignore_index=True,
)
comparison_df.to_csv(COMPARISON_PATH, index=False)

print(f"Updated: {COMPARISON_PATH}")
comparison_df

## 5. Comparison with Previous Models

In [ ]:
best_pr = comparison_df.sort_values("PR_AUC", ascending=False).iloc[0]
best_f1_row = comparison_df.sort_values("F1", ascending=False).iloc[0]

display(comparison_df.sort_values("PR_AUC", ascending=False))

print(f"\nBest model by PR-AUC: {best_pr['Model']} ({best_pr['PR_AUC']})")
print(f"Best model by F1: {best_f1_row['Model']} ({best_f1_row['F1']})")
print(f"Predicted churners (XGBoost, threshold=0.5): {xgb_metrics['Predicted_Churners']}")

print("\nTakeaways:")
print("- XGBoost improves PR-AUC over Random Forest and Logistic Regression models.")
print("- Random Forest may still lead on F1 at the default 0.5 threshold.")
print("- XGBoost offers strong recall with competitive ranking metrics for churn.")
print("- Threshold optimization (later) may change the precision/recall trade-off.")

## Summary

- **Best XGBoost settings:** selected by validation F1 from a compact grid
- **`scale_pos_weight` baseline:** train negative/positive ratio
- **Test set:** not used

**Next step (not performed here):** Imbalance experiments (SMOTE, etc.) or model comparison summary.